In [ ]:
from time import time
from typing import List

from ctranslate2 import Translator
from datasets import load_dataset
from pydantic import DirectoryPath
from sacrebleu import corpus_bleu, corpus_chrf
from transformers import AutoTokenizer

In [ ]:
class OpusmtTranslator:
    def __init__(self, model_path: DirectoryPath, model_id: str, ct2_device: str = "auto", compute_type: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.translator = Translator(
            model_path, device=ct2_device, compute_type=compute_type)

    def tokenize(self, x: List[str]):
        return self.tokenizer.convert_ids_to_tokens(self.tokenizer.encode(x))

    def translate(self, x, beam_size: int = 5):
        if isinstance(x, str):
            return_string = True
            x = [x]
        else:
            return_string = False

        x = [self.tokenize(i) for i in x]
        ret = self.translator.translate_batch(
            x, max_batch_size=32, beam_size=beam_size)
        ret = [self.tokenizer.convert_tokens_to_string(
            i.hypotheses[0]) for i in ret]
        if return_string:
            return ret[0]
        else:
            return ret

In [ ]:
class NLLBTranslator:
    def __init__(self, model_path: DirectoryPath, model_id: str, src_lang: str, tgt_lang: str, ct2_device: str = "auto", compute_type: str = "auto"):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, src_lang=src_lang)
        self.translator = Translator(
            model_path, device=ct2_device, compute_type=compute_type)
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang

    def tokenize(self, x: List[str]):
        return self.tokenizer.convert_ids_to_tokens(self.tokenizer.encode(x))

    def translate(self, x: str | List[str], beam_size: int = 5):
        if isinstance(x, str):
            return_string = True
            x = [x]
        else:
            return_string = False

        x = [self.tokenize(i) for i in x]
        ret = self.translator.translate_batch(
            x, target_prefix=[[self.tgt_lang]]*len(x), max_batch_size=32, beam_size=5
        )
        ret = [self.tokenizer.convert_tokens_to_string(
            i.hypotheses[0][1:]) for i in ret]
        if return_string:
            return ret[0]
        else:
            return ret

## Flores Data

In [ ]:
src_lang_flores = "ukr_Cyrl"
tgt_lang_flores = "eng_Latn"

ct2_device = "cuda"
ct2_compute_type = "float16" if ct2_device == "cuda" else "auto"

tgt = load_dataset("openlanguagedata/flores_plus",
                   tgt_lang_flores, split="devtest")
src = load_dataset("openlanguagedata/flores_plus",
                   src_lang_flores, split="devtest")

src = [i['text'] for i in src]
tgt = [i['text'] for i in tgt]

## Opus-MT

In [ ]:
opusmt_model = "../ct2-models/opus-mt-uk-en-ct2/"
opusmt_id = "Helsinki-NLP/opus-mt-uk-en"
opusmt_model = "../ct2-models/opus-mt-zle-en-big-ct2"
opusmt_id = "Helsinki-NLP/opus-mt-tc-big-zle-en"

t = OpusmtTranslator(opusmt_model, opusmt_id,
                     ct2_device=ct2_device, compute_type=ct2_compute_type)

In [ ]:
t1 = time()
mt = t.translate(src, beam_size=5)
t2 = time()
del t
opus_time = t2 - t1

In [ ]:
opus_bleu = corpus_bleu(mt, [tgt]).score

In [ ]:
opus_chrf = corpus_chrf(mt, [tgt]).score

## NLLB

In [ ]:
t = NLLBTranslator("../ct2-models/nllb-200-distilled-1.3B-ct2/",
                   "facebook/nllb-200-distilled-1.3B", src_lang_flores, tgt_lang_flores, ct2_device, compute_type=ct2_compute_type)

In [ ]:
t1 = time()
mt = t.translate(src, beam_size=5)
t2 = time()
del t
nllb_time = t2 - t1
nllb_time

In [ ]:
nllb_bleu = corpus_bleu(mt, [tgt]).score
nllb_bleu

In [ ]:
nllb_chrf = corpus_chrf(mt, [tgt]).score
nllb_chrf

## QuickMT

In [ ]:
from quickmt import Translator

In [ ]:
t = Translator("quickmt/quickmt-uk-en",
               device=ct2_device, compute_type=ct2_compute_type)

In [ ]:
t = Translator("/home/mark/mt/quickmt-models/quickmt-uk-en",
               device=ct2_device, compute_type=ct2_compute_type)

In [ ]:
src[1]

In [ ]:
t(src[1], beam_size=5)

In [ ]:
t(src[1], sampling_temperature=1.2, beam_size=1, sampling_topk=50, sampling_topp=0.9)

In [ ]:
t1 = time()
mt = t.translate(src, beam_size=5, max_batch_size=32)
t2 = time()
del t
quickmt_time = t2 - t1
quickmt_time

In [ ]:
quickmt_bleu = corpus_bleu(mt, [tgt]).score
quickmt_bleu

In [ ]:
quickmt_chrf = corpus_chrf(mt, [tgt]).score
quickmt_chrf

## Results

Print out markdown table

In [ ]:
print(f"| model | time | bleu | chrf |")
print(f"| ----------- | ------- | ------ | -------|")
print(f"| {opusmt_id} | {(opus_time):.2f} | {
      opus_bleu:.2f}| {opus_chrf:.2f} |")
print(f"| quickmt_uk_en | {(quickmt_time):.2f} | {
      quickmt_bleu:.2f}| {quickmt_chrf:.2f} |")
print(f"| facebook/nllb-200-distilled-1.3B | {
      (nllb_time):.2f} | {nllb_bleu:.2f}| {nllb_chrf:.2f} |")

| model | time | bleu | chrf |
| ----------- | ------- | ------ | -------|
| quickmt_uk_en | 1.12 | 39.88| 65.65 |
| Helsinki-NLP/opus-mt-tc-big-zle-en | 3.03 | 39.07| 64.98 |
| facebook/nllb-200-distilled-1.3B | 17.25 | 33.59| 63.50 |
| CohereLabs/tiny-aya-global | 23.36 | 37.60| 63.73 |
| google/gemma-4-E2B-it | 47.71 | 37.22| 63.75 |


## Benchmark LLMs running With vLLM

```bash
# Create env
conda create -n vllm python=3.12 pip
conda activate vllm
pip install vllm openai tqdm huggingface_hub

# May be required to run latest models:
pip install --upgrade transformers

# Log in, in case model is gated
hf auth login

# Launch LLM
vllm serve CohereLabs/tiny-aya-global --gpu_memory_utilization=0.85 --max_num_seqs=32 --max_model_len 4096
vllm serve google/gemma-4-E2B-it --gpu_memory_utilization=0.90 --max_num_seqs=8 --max_model_len 1024
```

In [ ]:
import asyncio
import os
import time
from concurrent.futures import ThreadPoolExecutor

import openai
from openai import OpenAI
from tqdm import tqdm
from tqdm.asyncio import tqdm_asyncio

client = OpenAI(api_key="", base_url="http://localhost:8000/v1", timeout=120)

model_id = client.models.list().data[0].id

In [ ]:
def translate_gemma(x, src_lang: str = "uk", tgt_lang: str = "en"):
    """Sends a single async request to the OpenAI API."""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                  "role": "user",
                  "content": f"<<<source>>>{src_lang}<<<target>>>{tgt_lang}<<<text>>>{x}"
                }
            ],
            temperature=0.0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return ""
        
def translate(x, src_lang="Ukranian", tgt_lang="English"):
    """Sends a single async request to the OpenAI API."""
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {
                    "role": "user",
                    "content": f"Translate the following into English, without commentary or explanation.\n\n{x}",
                    #"content": f"You are a professional {src_lang}-to-{tgt_lang} translator. Your goal is to accurately convey the meaning and nuances of the original {src_lang} text while adhering to {tgt_lang} grammar, vocabulary, and cultural sensitivities. Produce only the {tgt_lang} translation, without any additional explanations or commentary. Retain the paragraph breaks (double new lines) from the input text. Please translate the following {src_lang} text into {tgt_lang}:\n\n{x}"
                }
            ],
            temperature=0.0,
            max_tokens=512,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return ""

if model_id == "Infomaniak-AI/vllm-translategemma-4b-it":
    translate = translate_gemma


In [ ]:
translate_gemma("Test")

In [ ]:
def translate_parallel(items):
    with ThreadPoolExecutor(max_workers=64) as executor:
        results = list(tqdm(executor.map(translate, items), total=len(items)))
    return results

t1 = time.time()
mt = translate_parallel(src)
t2 = time.time()

In [ ]:
llm_time = t2 - t1
llm_bleu = corpus_bleu(mt, [tgt]).score
llm_chrf = corpus_chrf(mt, [tgt]).score

In [ ]:
print(f"| model | time | bleu | chrf |")
print(f"| ----------- | ------- | ------ | -------|")
print(f"| {model_id} | {(llm_time):.2f} | {
      llm_bleu:.2f}| {llm_chrf:.2f} |")